In [159]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


In [160]:
df = pd.read_csv("https://raw.githubusercontent.com/abdelrahmanhela/Smart-Used-Car-Selection-and-Pricing-System/refs/heads/main/Data%20Cleaning/cars_clean.csv")
df.head()

,id,make,model,year,km,price_egp,transmission,color,city,seller_type,fuel_type
0,7284417,Toyota,Yaris,2009,170000.0,540000,Automatic,Dark blue,Heliopolis,dealer,Petrol
1,7296785,Toyota,Camry,2013,113000.0,1000000,Automatic,White,Nasr city,private,Petrol
2,7293331,Toyota,Corolla,2024,95000.0,1380000,Automatic,Black,Mohandessin,private,Petrol
3,7288941,Toyota,Corolla,2008,210000.0,585000,Automatic,Black,Heliopolis,dealer,Petrol
4,7295928,Toyota,Corolla,2022,126000.0,980000,Automatic,Dark red,Maadi,dealer,Petrol


## Separate Features & Target

In [161]:
df['car_type'] = df['make'] + '_' + df['model'] 


In [162]:
counts = df['car_type'].value_counts()

print("Total rows:", len(df))
print("Rare classes:", (counts < 2).sum())
print("Rows in rare classes:", counts[counts < 2].sum())
print("Remaining rows:", counts[counts >= 2].sum())

Total rows: 3265
Rare classes: 33
Rows in rare classes: 33
Remaining rows: 3232


In [163]:
df_cls = df[df['car_type'].isin(counts[counts >= 2].index)].copy()

In [164]:
df_cls.isna().sum()

id                0
make              0
model             0
year              0
km              181
price_egp         0
transmission      0
color            54
city              0
seller_type       0
fuel_type         0
car_type          0
dtype: int64

In [165]:
X = df_cls.drop(columns=['make', 'model', 'car_type', 'id'])
y = df_cls['car_type']

In [166]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=45,
    stratify=y
)

In [167]:
color_mode = X_train['color'].mode()[0]

X_train['color'] = X_train['color'].fillna(color_mode)
X_test['color'] = X_test['color'].fillna(color_mode)

In [168]:
X_train['km'] = X_train['km'].fillna(X_train['km'].median())
X_test['km'] = X_test['km'].fillna(X_train['km'].median())
print(X_train['km'].isna().sum())
print(X_test['km'].isna().sum())

0
0


## Encoding

In [169]:
df_cls

,id,make,model,year,km,price_egp,transmission,color,city,seller_type,fuel_type,car_type
0,7284417,Toyota,Yaris,2009,170000.0,540000,Automatic,Dark blue,Heliopolis,dealer,Petrol,Toyota_Yaris
1,7296785,Toyota,Camry,2013,113000.0,1000000,Automatic,White,Nasr city,private,Petrol,Toyota_Camry
2,7293331,Toyota,Corolla,2024,95000.0,1380000,Automatic,Black,Mohandessin,private,Petrol,Toyota_Corolla
3,7288941,Toyota,Corolla,2008,210000.0,585000,Automatic,Black,Heliopolis,dealer,Petrol,Toyota_Corolla
4,7295928,Toyota,Corolla,2022,126000.0,980000,Automatic,Dark red,Maadi,dealer,Petrol,Toyota_Corolla
...,...,...,...,...,...,...,...,...,...,...,...,...
3260,7243761,Hyundai,Excel,1997,470000.0,85000,Manual,Red,Shubra al-Khaimah,private,Petrol,Hyundai_Excel
3261,7243621,Hyundai,Verna,2007,166000.0,330000,Manual,Dark red,Damanhur,private,Petrol,Hyundai_Verna
3262,7243617,Hyundai,Getz,2008,162000.0,470000,Automatic,Red,Talkha,private,Petrol,Hyundai_Getz
3263,7243499,Hyundai,Getz,2011,200000.0,425000,Automatic,Dark red,Mansoura,private,Petrol,Hyundai_Getz


In [170]:
ohe_cols = ['transmission', 'seller_type', 'fuel_type', 'color']
freq_cols = ['city']

**TARGET ENCODING**

In [171]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

y_train = pd.Series(
    y_train_encoded,
    index=y_train.index,
    name=y_train.name
)

y_test = pd.Series(
    y_test_encoded,
    index=y_test.index,
    name=y_test.name
)

**OHE**

In [172]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_ohe = ohe.fit_transform(X_train[ohe_cols])
X_test_ohe = ohe.transform(X_test[ohe_cols])

ohe_names = ohe.get_feature_names_out(ohe_cols)

X_train_ohe = pd.DataFrame(
    X_train_ohe,
    columns=ohe_names,
    index=X_train.index
)

X_test_ohe = pd.DataFrame(
    X_test_ohe,
    columns=ohe_names,
    index=X_test.index
)

In [173]:
X_train = X_train.drop(columns=ohe_cols)
X_test = X_test.drop(columns=ohe_cols)

X_train = pd.concat([X_train, X_train_ohe], axis=1)
X_test = pd.concat([X_test, X_test_ohe], axis=1)

In [174]:
X_train

,year,km,price_egp,city,transmission_Automatic,transmission_Manual,seller_type_dealer,seller_type_private,fuel_type_CNG,fuel_type_Diesel,...,color_Mocha,color_Olive,color_Orange,color_Other,color_Petroleum,color_Purple,color_Red,color_Silver,color_White,color_Yellow
2221,2021,50000.0,815000,Nasr city,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2492,2008,1.0,475000,Alexandria,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1793,2015,139000.0,740000,Beni Suef,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1597,2008,70000.0,420000,Alexandria Desert,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3126,2008,235000.0,420000,Cairo,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3154,2010,866000.0,290000,El Faiyum,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2018,2017,133900.0,1200000,Sohag,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,2008,22000.0,525000,Cairo,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2199,2021,4000.0,1675000,Nasr city,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


**Frequency**

In [175]:
for col in freq_cols:
    freq = X_train[col].value_counts(normalize=True)

    X_train[col] = X_train[col].map(freq)
    X_test[col] = X_test[col].map(freq).fillna(0)

In [176]:
X_train

,year,km,price_egp,city,transmission_Automatic,transmission_Manual,seller_type_dealer,seller_type_private,fuel_type_CNG,fuel_type_Diesel,...,color_Mocha,color_Olive,color_Orange,color_Other,color_Petroleum,color_Purple,color_Red,color_Silver,color_White,color_Yellow
2221,2021,50000.0,815000,0.075048,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2492,2008,1.0,475000,0.046809,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1793,2015,139000.0,740000,0.006963,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1597,2008,70000.0,420000,0.003868,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3126,2008,235000.0,420000,0.140812,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3154,2010,866000.0,290000,0.011605,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2018,2017,133900.0,1200000,0.008897,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,2008,22000.0,525000,0.140812,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2199,2021,4000.0,1675000,0.075048,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [177]:
print(X_train.head())
print(X_train.dtypes)

      year        km  price_egp      city  transmission_Automatic  \
2221  2021   50000.0     815000  0.075048                     1.0   
2492  2008       1.0     475000  0.046809                     0.0   
1793  2015  139000.0     740000  0.006963                     1.0   
1597  2008   70000.0     420000  0.003868                     1.0   
3126  2008  235000.0     420000  0.140812                     1.0   

      transmission_Manual  seller_type_dealer  seller_type_private  \
2221                  0.0                 0.0                  1.0   
2492                  1.0                 0.0                  1.0   
1793                  0.0                 0.0                  1.0   
1597                  0.0                 0.0                  1.0   
3126                  0.0                 0.0                  1.0   

      fuel_type_CNG  fuel_type_Diesel  ...  color_Mocha  color_Olive  \
2221            0.0               0.0  ...          0.0          0.0   
2492            0.0 

## Scaling

In [178]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scale_col = ['km', 'year', 'price_egp']

X_train[scale_col] = scaler.fit_transform(X_train[scale_col])
X_test[scale_col] = scaler.transform(X_test[scale_col])

In [179]:
print(X_train[['year', 'km', 'price_egp']].head())

          year        km  price_egp
2221  0.734697 -0.919789  -0.096323
2492 -0.945203 -1.415005  -0.545726
1793 -0.040642 -0.038286  -0.195456
1597 -0.945203 -0.721698  -0.618424
3126 -0.945203  0.912549  -0.618424


In [180]:
X_train.to_csv('X_train_cls.csv', index=False)
X_test.to_csv('X_test_cls.csv', index=False)

y_train.to_csv('y_train_cls.csv', index=False)
y_test.to_csv('y_test_cls.csv', index=False)